# Automatically Generating Bounding Boxes from FEN Notation

This notebook demonstrates how to parse a FEN string, generate bounding boxes for the chess pieces on a uniformly aligned chessboard image, and draw those boxes using OpenCV. This approach assumes that the board is cropped and the grid is uniform.

In [1]:
import cv2
import numpy as np
import os

In [2]:
# path to the training data directory
data_dir = "..\\archive\\train\\"
num_of_images = 1000
i=0
image_paths = []
for filename in os.listdir(data_dir):
    
    if i >= num_of_images:
        break
    
    if filename.endswith(".jpeg"):
        i += 1
        image_path = os.path.join(data_dir, filename)
        image_paths.append(image_path)
        continue

print(f'Read {len(image_paths)} images from {data_dir}')
# load first image and get dimensions
image_path = image_paths[0]
image = cv2.imread(image_path)
height, width, channels = image.shape



Read 1000 images from ..\archive\train\


### Create FEN board

In [3]:
boards = []
for i, image_path in enumerate(image_paths):
    # get FEN notation string from filename of image_path
    fen = os.path.basename(image_path).split('.')[0].replace('_', ' ').strip()
    print(f'FEN: {fen}')

    if '-' in fen and '/' not in fen:
        ranks = fen.split('-')
    else:
        ranks = fen.split('/')
    board = []
    for rank in ranks:
        row = []
        for char in rank:
            if char.isdigit():
                # Expand the digit to empty squares
                row.extend([""] * int(char))
            else:
                row.append(char)
        board.append(row)
    boards.append(board)

print(f'Read {len(boards)} boards from images.')

FEN: 1b1B1b2-2pK2q1-4p1rB-7k-8-8-3B4-3rb3
FEN: 1b1b1b2-3r4-1rK4b-R7-R2R1k2-2Bp4-2P5-2r5
FEN: 1B1B1K2-3p1N2-6k1-R7-5P2-4q3-7R-1B6
FEN: 1b1B1K2-R2B4-7P-3b4-3R2B1-8-3R4-4Qk2
FEN: 1b1b1n2-1K1RN1b1-3pbN2-8-4q1k1-4P3-8-2n3N1
FEN: 1B1B1N2-1r6-n2R2k1-7b-1B6-8-8-Kn6
FEN: 1B1b1Nn1-8-3p4-2K5-8-B5P1-p7-4k1nb
FEN: 1B1b1R2-1b2R3-p7-3N4-4b3-n1BP1k2-2K5-6q1
FEN: 1B1b1R2-R4r2-3Q4-6p1-3B1R1q-3Knk2-1P6-8
FEN: 1B1b1Rr1-1p1n4-8-N1K3N1-3n4-P2k4-3p4-2N5
FEN: 1B1b2b1-5P2-2n5-1B2P3-5p2-6n1-1B4k1-2RRK3
FEN: 1b1b2Bk-5n2-K7-1N1B4-2R5-p7-2BB4-4r3
FEN: 1b1b2Bq-5N2-2R5-8-7k-p2K3P-2N1Pq2-6R1
FEN: 1b1b2N1-1P1r4-2p5-5R2-6p1-3p1P2-6k1-K7
FEN: 1b1B2n1-3K1PN1-B6N-3k1B2-5p2-8-1B3q2-r1b5
FEN: 1b1b2R1-2n2p2-8-4n1k1-6rp-2P5-6P1-2KQ3n
FEN: 1b1b2R1-6Q1-K1qkQ1n1-1n6-2P2r2-5r2-8-7r
FEN: 1b1b2RR-4p3-6B1-2R5-4N1N1-1q1K4-2PR1b2-4k3
FEN: 1b1B3b-6Q1-Q7-2N5-4bn2-5k1n-3K4-8
FEN: 1b1B3n-2P2r2-5N2-3n1N2-P4n2-8-8-1K4k1
FEN: 1B1b3n-8-2Rpk1Qb-2K2r2-P7-5b2-r7-1b4N1
FEN: 1b1b3r-2P1b3-2n5-N7-1B1k4-K3n1pb-7b-n7
FEN: 1b1B3r-6k1-8-5P1p-1K6-8-8-3n1

In [4]:
height, width = image.shape[:2]
cell_width = width / 8
cell_height = height / 8

num_boxes = 0

boxes = []
for num, board in enumerate(boards):
    board_boxes = {'image_path':image_paths[num], 'pieces': []}
    for i, row in enumerate(board):
        for j, piece in enumerate(row):
            if piece != "":
                x_min = int(j * cell_width)
                y_min = int(i * cell_height)
                x_max = int((j + 1) * cell_width)
                y_max = int((i + 1) * cell_height)
                board_boxes['pieces'].append({
                    'piece': piece,
                    'bbox': (x_min, y_min, x_max, y_max),
                    'row': i,
                    'col': j
                })
                num_boxes += len(board_boxes)
                
    boxes.append(board_boxes)
    
print(boxes[:5])
    
print(f'Read a total of {num_boxes} boxes from {len(boards)} board images.')

[{'image_path': '..\\archive\\train\\1b1B1b2-2pK2q1-4p1rB-7k-8-8-3B4-3rb3.jpeg', 'pieces': [{'piece': 'b', 'bbox': (50, 0, 100, 50), 'row': 0, 'col': 1}, {'piece': 'B', 'bbox': (150, 0, 200, 50), 'row': 0, 'col': 3}, {'piece': 'b', 'bbox': (250, 0, 300, 50), 'row': 0, 'col': 5}, {'piece': 'p', 'bbox': (100, 50, 150, 100), 'row': 1, 'col': 2}, {'piece': 'K', 'bbox': (150, 50, 200, 100), 'row': 1, 'col': 3}, {'piece': 'q', 'bbox': (300, 50, 350, 100), 'row': 1, 'col': 6}, {'piece': 'p', 'bbox': (200, 100, 250, 150), 'row': 2, 'col': 4}, {'piece': 'r', 'bbox': (300, 100, 350, 150), 'row': 2, 'col': 6}, {'piece': 'B', 'bbox': (350, 100, 400, 150), 'row': 2, 'col': 7}, {'piece': 'k', 'bbox': (350, 150, 400, 200), 'row': 3, 'col': 7}, {'piece': 'B', 'bbox': (150, 300, 200, 350), 'row': 6, 'col': 3}, {'piece': 'r', 'bbox': (150, 350, 200, 400), 'row': 7, 'col': 3}, {'piece': 'b', 'bbox': (200, 350, 250, 400), 'row': 7, 'col': 4}]}, {'image_path': '..\\archive\\train\\1b1b1b2-3r4-1rK4b-R7-R2R1

# Augment board images by size and noise

In [5]:
import random
def create_random_background(board_w, board_h, min_scale=1.1, max_scale=2.0):
    """
    Create a random background image.
    The background dimensions are based on the board dimensions scaled by a random factor.
    The background is given a base gray color with added random noise.
    """
    scale = random.uniform(min_scale, max_scale)
    # Randomly choose vertical or horizontal orientation:
    if random.choice([True, False]):
        bg_w = int(board_w * scale)
        bg_h = int(board_h * scale * random.uniform(1.2, 1.8))
    else:
        bg_w = int(board_w * scale * random.uniform(1.2, 1.8))
        bg_h = int(board_h * scale)
    
    
    background_style = np.random.choice(['light', 'dark', 'noise', 'noise', 'noise'])  # choose light or dark background
    
    if background_style == 'light':
        background = np.full((bg_h, bg_w, 3), np.random.randint(150, 230), dtype=np.uint8)  # light gray base
    elif background_style == 'dark':
        background = np.full((bg_h, bg_w, 3), np.random.randint(0, 150), dtype=np.uint8)  # dark gray base
    elif background_style == 'noise':
        background = np.full((bg_h, bg_w, 3), np.random.randint(0, 230), dtype=np.uint8)  # light gray base
        noise = np.random.randint(0, 100, (bg_h, bg_w, 3), dtype=np.uint8)
        background = cv2.add(background, noise)
    return background

def augment_board_image(board_img, piece_boxes):
    """
    Augment a cropped chessboard image by randomly scaling it, adding noise via a random background,
    and placing it at a random location on that background.
    
    Parameters:
      board_img: The cropped chessboard image (numpy.ndarray).
      piece_boxes: List of piece boxes (each with key 'bbox') defined relative to board_img.
    
    Returns:
      composite_img: The new composite image.
      new_board_box: The bounding box (x_min, y_min, x_max, y_max) of the board in the composite image.
      new_piece_boxes: The updated piece boxes (with bboxes adjusted to the composite image coordinates).
    """
    
    orig_h, orig_w = board_img.shape[:2]
    
    # Apply random scaling to the board
    scale_factor = random.uniform(0.9, 1.1)
    new_w = int(orig_w * scale_factor)
    new_h = int(orig_h * scale_factor)
    board_scaled = cv2.resize(board_img, (new_w, new_h))
    
    # Create a random background based on scaled board dimensions
    background = create_random_background(new_w, new_h)
    background = add_random_overlays(background)

    bg_h, bg_w = background.shape[:2]
    
    # Randomly determine the board's top-left position on the background (ensure full board is visible)
    max_x = bg_w - new_w
    max_y = bg_h - new_h
    top_left_x = random.randint(0, max(0, max_x))
    top_left_y = random.randint(0, max(0, max_y))
    
    # Create composite image
    composite_img = background.copy()
    composite_img[top_left_y:top_left_y+new_h, top_left_x:top_left_x+new_w] = board_scaled
    
    # New bounding box for the board in the composite image:
    new_board_box = (top_left_x, top_left_y, top_left_x + new_w, top_left_y + new_h)
    
    # Update piece boxes: scale coordinates and add the board offset
    new_piece_boxes = []
    for box in piece_boxes:
        piece = box['piece']
        (x_min, y_min, x_max, y_max) = box['bbox']
        new_x_min = int(x_min * scale_factor) + top_left_x
        new_y_min = int(y_min * scale_factor) + top_left_y
        new_x_max = int(x_max * scale_factor) + top_left_x
        new_y_max = int(y_max * scale_factor) + top_left_y
        new_piece_boxes.append({
            'piece': piece,
            'bbox': (new_x_min, new_y_min, new_x_max, new_y_max)
        })
    
    new_piece_boxes.append({
        'piece': 'board',
        'bbox': new_board_box
    })
    
    return composite_img, new_piece_boxes

import glob

def add_random_overlays(image, overlay_images_folder="sample_profile_pictures",
                        add_timer=True, add_small_images=True,
                        timer_chance=0.9, small_image_chance=0.85):
    """
    Adds random overlays to the image to simulate UI elements such as timers and profile pictures.
    
    Parameters:
      image (numpy.ndarray): Input image (BGR).
      overlay_images_folder (str): Path to a folder containing small images to overlay.
      add_timer (bool): Whether to add a timer text overlay.
      add_small_images (bool): Whether to overlay small images.
      timer_chance (float): Probability to add timer text.
      small_image_chance (float): Probability to add a small image overlay.
      
    Returns:
      augmented_image (numpy.ndarray): The image with added overlays.
    """
    augmented_image = image.copy()
    h, w = augmented_image.shape[:2]
    
    # --- Add Timer Text Overlay ---
    if add_timer and random.random() < timer_chance:
        for x in range(10):
            # Create a random timer string (MM:SS)
            minutes = random.randint(0, 59)
            seconds = random.randint(0, 59)
            timer_text = f"{minutes:02d}:{seconds:02d}"
            
            # Randomly choose a position (ensure text fits inside the image)
            text_width, text_height = 100, 30  # approximate; could be refined
            pos_x = random.randint(0, max(0, w - text_width))
            pos_y = random.randint(text_height, h)
            
            # Choose random font parameters
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = random.uniform(0.8, 1.2)
            thickness = random.randint(1, 3)
            # You can use a fixed color or randomize:
            text_color = random.randint(0,255)
            color = (text_color, text_color, text_color)
            # draw a recant rectangle around the text with alpha blending
            brightness_val = np.random.randint(0,255)
            overlay_color = (brightness_val,brightness_val,brightness_val)  # white color
            cv2.rectangle(augmented_image, (pos_x, pos_y-text_height), (pos_x + text_width, pos_y + text_height),
                        overlay_color, -1, cv2.LINE_AA)
            cv2.putText(augmented_image, timer_text, (pos_x, pos_y), font,
                        font_scale, color, thickness, cv2.LINE_AA)
        
            # add random chess position text
            piece = random.choice(["P", "N", "B", "R", "Q", "K", "p", "n", "b", "r", "q", "k", 'Nx', 'Bx', 'Rx', 'Qx', 'Kx', 'Bx', 'Rx', 'Qx', 'Kx'])
            square = random.choice(["a1", "b1", "c1", "d1", "e1", "f1", "g1", "h1", "a2", "b2", "c2", "d2", "e2", "f2", "g2", "h2", "a3", "b3", "c3", "d3", "e3", "f3", "g3", "h3", "a4", "b4", "c4", "d4", "e4", "f4", "g4", "h4"]) 
            text_to_put = f"{piece}{square}"
            pos_x = random.randint(0, max(0, w - text_width))
            pos_y = random.randint(text_height, h)
            # add text to the image
            text_color = random.randint(0,255)
            cv2.putText(augmented_image, text_to_put, (pos_x, pos_y), fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=1.5, color=(text_color, text_color, text_color), thickness=3)
        
    # --- Add Small Image Overlay ---
    if add_small_images and overlay_images_folder is not None and random.random() < small_image_chance:
        for x in range(10):
            # Get a list of small image file paths from the folder.
            small_image_paths = glob.glob(os.path.join(overlay_images_folder, "*"))
            if small_image_paths:
                chosen_path = random.choice(small_image_paths)
                # Use IMREAD_UNCHANGED to try to preserve alpha if available.
                small_img = cv2.imread(chosen_path, cv2.IMREAD_UNCHANGED)
                if small_img is not None:
                    # Randomly scale the small image (e.g., 20% to 60% of original size)
                    scale = random.uniform(0.05, 0.4)
                    orig_small_h, orig_small_w = small_img.shape[:2]
                    new_small_w = int(orig_small_w * scale)
                    new_small_h = int(orig_small_h * scale)
                    small_img = cv2.resize(small_img, (new_small_w, new_small_h))
                    
                    # Randomly choose a position on the image to place the small image.
                    max_x = w - new_small_w
                    max_y = h - new_small_h
                    pos_x = random.randint(0, max(0, max_x))
                    pos_y = random.randint(0, max(0, max_y))
                    
                    # Overlay the small image onto the augmented image.
                    # Check if small_img has an alpha channel.
                    if small_img.shape[2] == 4:
                        # Separate color and alpha channels
                        overlay = small_img[:,:,:3]
                        alpha_mask = small_img[:,:,3] / 255.0
                        # Region of interest in the augmented image.
                        roi = augmented_image[pos_y:pos_y+new_small_h, pos_x:pos_x+new_small_w]
                        # Blend the overlay with the ROI using the alpha mask.
                        for c in range(3):
                            roi[:,:,c] = roi[:,:,c] * (1 - alpha_mask) + overlay[:,:,c] * alpha_mask
                        augmented_image[pos_y:pos_y+new_small_h, pos_x:pos_x+new_small_w] = roi
                    else:
                        # If no alpha channel, blend with a fixed opacity (e.g., 50%).
                        augmented_image[pos_y:pos_y+new_small_h, pos_x:pos_x+new_small_w] = small_img
    
    return augmented_image


In [6]:
augmented_data = []  # list to hold new augmented entries
for board_entry in boxes:
    board_path = board_entry['image_path']
    print(f"Processing {board_path}...")
    board_img = cv2.imread(board_path)
    if board_img is None:
        print(f"Warning: Could not load {board_path}. Skipping.")
        continue
    
    # set a random low percentage of boards to not augment
    if random.random() < 0.1:
        print(f"Skipping augmentation for {board_path}.")
        # set board coordinates to the four image corners
        # create new_board_box_in_yolo_bbox_format xmin ymin xmax ymax
        new_board_box = (1, 1, board_img.shape[1]-1, board_img.shape[0]-1)  # xmin ymin xmax ymax

        augmented_entry = {
            'image_path': board_path,
            'image': board_img,
            'pieces': board_entry['pieces']
        }
        augmented_entry['pieces'].append({'piece':'board', 'bbox': new_board_box})
        augmented_data.append(augmented_entry)
        continue
    else:
        composite_img, new_piece_boxes = augment_board_image(board_img, board_entry['pieces'])

    # convert composite_img into a RGB image (OpenCV uses BGR by default)
    composite_img_rgb = cv2.cvtColor(composite_img, cv2.COLOR_BGR2RGB)
    
    augmented_entry = {
        'image_path': board_path,
        'image': composite_img_rgb,
        'pieces': new_piece_boxes
    }
    augmented_data.append(augmented_entry)

# # For debugging, print the first few augmented entries:
# for entry in augmented_data[:5]:
#     # show all the images in a window
#     cv2.imshow('Augmented Image', entry['image'])
#     cv2.waitKey(0)
#     cv2.destroyAllWindows()

boxes = augmented_data

# display the first few augmented entries
# for entry in boxes[:5]:
#     # show all the images in a window
#     cv2.imshow('Augmented Image', entry['image'])
#     cv2.waitKey(0)
#     cv2.destroyAllWindows()



Processing ..\archive\train\1b1B1b2-2pK2q1-4p1rB-7k-8-8-3B4-3rb3.jpeg...
Processing ..\archive\train\1b1b1b2-3r4-1rK4b-R7-R2R1k2-2Bp4-2P5-2r5.jpeg...
Processing ..\archive\train\1B1B1K2-3p1N2-6k1-R7-5P2-4q3-7R-1B6.jpeg...
Processing ..\archive\train\1b1B1K2-R2B4-7P-3b4-3R2B1-8-3R4-4Qk2.jpeg...
Processing ..\archive\train\1b1b1n2-1K1RN1b1-3pbN2-8-4q1k1-4P3-8-2n3N1.jpeg...
Processing ..\archive\train\1B1B1N2-1r6-n2R2k1-7b-1B6-8-8-Kn6.jpeg...
Skipping augmentation for ..\archive\train\1B1B1N2-1r6-n2R2k1-7b-1B6-8-8-Kn6.jpeg.
Processing ..\archive\train\1B1b1Nn1-8-3p4-2K5-8-B5P1-p7-4k1nb.jpeg...
Processing ..\archive\train\1B1b1R2-1b2R3-p7-3N4-4b3-n1BP1k2-2K5-6q1.jpeg...
Processing ..\archive\train\1B1b1R2-R4r2-3Q4-6p1-3B1R1q-3Knk2-1P6-8.jpeg...
Processing ..\archive\train\1B1b1Rr1-1p1n4-8-N1K3N1-3n4-P2k4-3p4-2N5.jpeg...
Processing ..\archive\train\1B1b2b1-5P2-2n5-1B2P3-5p2-6n1-1B4k1-2RRK3.jpeg...
Processing ..\archive\train\1b1b2Bk-5n2-K7-1N1B4-2R5-p7-2BB4-4r3.jpeg...
Processing ..\archive

## Draw Boxes

In [7]:
# read the image and draw boxes for first five images with opencv

for i in range(5):
    img = boxes[i]['image'].copy()
    for box in boxes[i]['pieces']:
        print(box['bbox'])
        x_min, y_min, x_max, y_max = box['bbox']
        cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
        cv2.putText(img, box['piece'], (x_min, y_min+25), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
        
    cv2.imshow(f'Box {i+1}', img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()



(80, 237, 130, 287)
(181, 237, 231, 287)
(282, 237, 332, 287)
(130, 287, 181, 337)
(181, 287, 231, 337)
(332, 287, 383, 337)
(231, 337, 282, 388)
(332, 337, 383, 388)
(383, 337, 433, 388)
(383, 388, 433, 438)
(181, 539, 231, 590)
(181, 590, 231, 640)
(231, 590, 282, 640)
(30, 237, 433, 640)
(97, 427, 144, 473)
(191, 427, 238, 473)
(285, 427, 332, 473)
(191, 473, 238, 520)
(97, 520, 144, 567)
(144, 520, 191, 567)
(379, 520, 426, 567)
(51, 567, 97, 614)
(51, 614, 97, 661)
(191, 614, 238, 661)
(285, 614, 332, 661)
(144, 661, 191, 708)
(191, 661, 238, 708)
(144, 708, 191, 755)
(144, 755, 191, 802)
(51, 427, 426, 802)
(230, 621, 280, 671)
(330, 621, 380, 671)
(430, 621, 480, 671)
(330, 671, 380, 721)
(430, 671, 480, 721)
(480, 721, 530, 771)
(180, 771, 230, 821)
(430, 821, 480, 871)
(380, 871, 430, 921)
(530, 921, 581, 971)
(230, 971, 280, 1022)
(180, 621, 581, 1022)
(70, 73, 118, 121)
(166, 73, 214, 121)
(262, 73, 310, 121)
(22, 121, 70, 169)
(166, 121, 214, 169)
(358, 169, 406, 217)
(166,

# Write bounding boxes to text files for YOLO training

In [8]:

class_mapping = {
    'K': 0,  # white king
    'Q': 1,  # white queen
    'R': 2,  # white rook
    'B': 3,  # white bishop
    'N': 4,  # white knight
    'P': 5,  # white pawn
    'k': 6,  # black king
    'q': 7,  # black queen
    'r': 8,  # black rook
    'b': 9,  # black bishop
    'n': 10, # black knight
    'p': 11,  # black pawn
    'board': 12 # board itself
}

# dictionary to keep count of the number of boxes per class
class_counts = {key: 0 for key in class_mapping.keys()}

# dictionary to keep track of the total number of boxes
total_boxes = 0

output_label_location = "..\\piece_detection"

for board in boxes:
    image = board['image']
    height, width = image.shape[:2]

    output_txt_path = os.path.join(output_label_location, f"{os.path.splitext(os.path.basename(board['image_path']))[0]}.txt")

    board['output_label_path'] = output_txt_path
    
    with open(output_txt_path, 'w') as f:
        for box in board['pieces']:
            piece = box['piece']

            class_counts[piece] += 1
            total_boxes += 1

            bbox = box['bbox']  # Format: (x_min, y_min, x_max, y_max)
            x_min, y_min, x_max, y_max = bbox
            # Compute the center of the bounding box
            x_center = ((x_min + x_max) / 2) / width
            y_center = ((y_min + y_max) / 2) / height
            # Compute width and height of the bounding box
            bbox_width = (x_max - x_min) / width
            bbox_height = (y_max - y_min) / height
            
            # Get the class id from the mapping.
            if piece not in class_mapping:
                print(f"Warning: Piece '{piece}' not found in class mapping. Skipping.")
                continue
            class_id = class_mapping[piece]
            
            # Write the YOLO-formatted annotation to file.
            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {bbox_width:.6f} {bbox_height:.6f}\n")

# show class distrubution of pieces
print(class_counts)



{'K': 1000, 'Q': 493, 'R': 902, 'B': 1439, 'N': 930, 'P': 929, 'k': 1000, 'q': 457, 'r': 944, 'b': 1455, 'n': 969, 'p': 960, 'board': 1000}


In [11]:

for board in boxes[:3]:
    img = board('image')
    original_img_path = board('image_path')
    
    # get corresponding bounding box coordinates from text file
    with open(os.path.join(output_label_location, os.path.splitext(os.path.basename(original_img_path))[0] + '.txt'), 'r') as f:
        lines = f.readlines()
        print(lines)
    
    # convert normalized coordinates to pixel coordinates
    for line in lines:
        parts = line.strip().split(' ')
        class_id = int(parts[0])
        x_center = float(parts[1]) * width
        y_center = float(parts[2]) * height
        bbox_width = float(parts[3]) * width
        bbox_height = float(parts[4]) * height
        print(f"Class: {class_id}, Center: ({x_center:.6f}, {y_center:.6f}), Width: {bbox_width:.6f}, Height: {bbox_height:.6f}")

    # draw opencv rectangles for bounding boxes on the image
    for line in lines:
        parts = line.strip().split(' ')
        class_id = int(parts[0])
        x_center = float(parts[1]) * width
        y_center = float(parts[2]) * height
        bbox_width = float(parts[3]) * width
        bbox_height = float(parts[4]) * height
        cv2.rectangle(img, (int(x_center - bbox_width / 2), int(y_center - bbox_height / 2)), (int(x_center + bbox_width / 2), int(y_center + bbox_height / 2)), (0, 255, 0), 2)
        cv2.putText(img, f"{class_id}", (int(x_center - bbox_width / 2) + 10, int(y_center - bbox_height / 2) + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.imshow('Image with Bounding Boxes', img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

TypeError: 'dict' object is not callable

In [9]:
# create YOLO formatted folder structure to train model 
import shutil
import random

images_train_dir = '..\\YOLO_detection\\images\\train'
images_val_dir = '..\\YOLO_detection\\images\\val'
labels_train_dir = '..\\YOLO_detection\\labels\\train'
labels_val_dir = '..\\YOLO_detection\\labels\\val'

os.makedirs(images_train_dir, exist_ok=True)
os.makedirs(images_val_dir, exist_ok=True)
os.makedirs(labels_train_dir, exist_ok=True)
os.makedirs(labels_val_dir, exist_ok=True)

train_percentage = 0.8
val_percentage = 1-train_percentage

# randomly select images for training and validation sets
labels = os.listdir(output_label_location)

for i, img in enumerate(labels):
    if os.path.splitext(img)[1] != '.txt':
        labels.pop(i)


random.shuffle(labels)
num_images = len(labels)
train_size = int(num_images * train_percentage)
val_size = num_images - train_size

print(f"Number of images: {num_images}")
print(f"Training size: {train_size}")
print(f"Validation size: {val_size}")

# split the images into training and validation sets
labels_train = labels[:train_size]
labels_val = labels[train_size:]

for label in labels_train:

    for board in boxes:
        if board['output_label_path'] == os.path.join(output_label_location, label):
            img = board['image']
            break

    label_path = os.path.join(output_label_location, label)
    output_image_path = os.path.join(images_train_dir, os.path.splitext(label)[0] + '.jpeg')
    
    shutil.move(label_path, labels_train_dir)
    cv2.imwrite(output_image_path, img)

for label in labels_val:
    
    for board in boxes:
        if board['output_label_path'] == os.path.join(output_label_location, label):
            img = board['image']
            break
        
    label_path = os.path.join(output_label_location, label)
    output_image_path = os.path.join(images_val_dir, os.path.splitext(label)[0] + '.jpeg')
    
    shutil.move(label_path, labels_val_dir)
    cv2.imwrite(output_image_path, img)

    


Number of images: 1000
Training size: 800
Validation size: 200
